In [ ]:
import pandas as pd
import re, math

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

sw_sastrawi = set(StopWordRemoverFactory().get_stop_words())

In [ ]:
# Load data
CSV_PATH = '../vaksin_campak.csv'
df = pd.read_csv(CSV_PATH, sep=';')

print('Jumlah data: ', df.shape)
df.head(3)

In [ ]:
def text_preprocessing(text):
    # Casefolding
    cleaned_text = text.lower()
    
    # Noise removal
    cleaned_text = re.sub(r'@\w+|#\w+', '', cleaned_text)
    cleaned_text = cleaned_text.encode('ascii', 'ignore').decode('ascii')
    cleaned_text = re.sub(r'[0-9]+', '', cleaned_text)
    cleaned_text = re.sub(r'[^\w\s]', ' ', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

    raw_tokens = cleaned_text.split()

    filtered_tokens = [t for t in raw_tokens if t not in sw_sastrawi and len(t) >= 3]

    tokens =  [stemmer.stem(t) for t in filtered_tokens]

    final_text = ' '.join(tokens)

    return cleaned_text, final_text, tokens

df[['cleaned_text', 'final_text', 'tokens']] = df['komentar'].apply(
    lambda x: pd.Series(text_preprocessing(x))
)

pd.set_option('display.max_colwidth', None)
display(df[['komentar', 'cleaned_text', 'final_text', 'tokens']].head(3))

In [ ]:
def build_inverted_index(df):
    inverted_index = {}

    for doc_id, row in df.iterrows():
        tokens = row['tokens']
        
        if not tokens:
            continue

        for token in tokens:
            if token not in inverted_index:
                inverted_index[token] = {}
            inverted_index[token][str(doc_id)] = inverted_index[token].get(str(doc_id), 0) + 1
    return inverted_index

inverted_index = build_inverted_index(df)

# Preview inverted index
print(f"Jumlah kata unik di Inverted Index: {len(inverted_index)}")
print()
for word in list(inverted_index.keys())[:5]:
    print(f"Kata: '{word}' -> Muncul di: {inverted_index[word]}")

In [ ]:
def calculated_tfidf(inverted_index: dict, total_docs: int):
    tfidf_index = {}
    idf_dict = {}
    doc_length_sq = {}

    for term, doc_dict in inverted_index.items():
        df_t = len(doc_dict)
        idf = math.log10((total_docs + 1) / (df_t + 1)) + 1
        idf_dict[term] = idf
        
        tfidf_index[term] = {}
        for doc_id, raw_tf in doc_dict.items():
            log_tf = (1 + math.log10(raw_tf)) if raw_tf > 0 else 0
            weight = log_tf * idf
            tfidf_index[term][doc_id] = weight

            doc_length_sq[doc_id] = doc_length_sq.get(doc_id, 0) + weight ** 2

    doc_magnitudes = {
        doc_id: math.sqrt(sq)
        for doc_id, sq in doc_length_sq.items()
    }
    return tfidf_index, idf_dict, doc_magnitudes

# Calculate TF-IDF
total_docs = len(df)
tfidf_index, idf_dict, doc_magnitudes = calculated_tfidf(inverted_index, total_docs)

# Create TF-IDF DataFrame
tfidf_matrix = {}
for term, doc_dict in tfidf_index.items():
    for doc_id, tfidf_value in doc_dict.items():
        if doc_id not in tfidf_matrix:
            tfidf_matrix[doc_id] = {}
        tfidf_matrix[doc_id][term] = tfidf_value

# Convert to DataFrame
df_tfidf = pd.DataFrame(tfidf_matrix).fillna(0).T

# Sort
df_tfidf = df_tfidf.sort_index(key=lambda x: x.astype(int))
df_tfidf.index = [f'Komentar {int(idx) + 1}' for idx in df_tfidf.index]

display(df_tfidf.head(5))

In [ ]:
def search(query: str, tfidf_index: dict, idf_dict: dict, doc_magnitudes: dict, doc_lookup: dict):
    _, _, query_tokens = text_preprocessing(query)
    if not query_tokens:
        return []
    
    query_raw_tf = {}
    for token in query_tokens:
        query_raw_tf[token] = query_raw_tf.get(token, 0) + 1

    query_tfidf = {}
    query_length_sq = 0.0

    for term, raw_tf in query_raw_tf.items():
        if term not in idf_dict:
            continue
        log_tf = (1 + math.log10(raw_tf)) if raw_tf > 0 else 0
        weight = log_tf * idf_dict[term]
        query_tfidf[term] = weight
        query_length_sq += weight ** 2

    if query_length_sq == 0:
        return []
    
    query_magnitude = math.sqrt(query_length_sq)

    dot_products = {}
    for term, q_weight in query_tfidf.items():
        if term in tfidf_index:
            for doc_id, d_weight in tfidf_index[term].items():
                dot_products[doc_id] = dot_products.get(doc_id, 0) + (q_weight * d_weight)

    if not dot_products:
        return []

    scores = []
    for doc_id, dot_prod in dot_products.items():
        denom = query_magnitude * doc_magnitudes.get(doc_id, 1.0)
        if denom == 0:
            continue
        similarity = dot_prod / denom
        
        row_data = doc_lookup[doc_id]
        scores.append({
            "doc_id": doc_id,
            "komentar": row_data['komentar'],
            "score": round(similarity, 4)
        })

    scores = sorted(scores, key=lambda x: x['score'], reverse=True)
    return scores

# Create doc_lookup
doc_lookup = {}
for doc_id, row in df.iterrows():
    doc_lookup[str(doc_id)] = {
        'komentar': row['komentar']
    }

# Test search
query = "vaksin campak"
results = search(query, tfidf_index, idf_dict, doc_magnitudes, doc_lookup)

print(f"Query: '{query}'")
print(f"Jumlah hasil: {len(results)}\n")

for i, result in enumerate(results[:5], 1):
    print(f"{i}. Doc {result['doc_id']} (Score: {result['score']})")
    print(f"   Komentar: {result['komentar'][:100]}...")
    print()